# Hands-on: Bayesian optimization of synthetic reaction yield

**Question:** Which reaction temperature should we test next, given a small experimental budget?

You will optimise a synthetic (not real experimental) reaction response between **60 C and 140 C**. The same objective is used in the interactive demos shown on slides. Start with three noisy observations, fit a Gaussian process, compute UCB and run sequential experimental decisions.

**Colab instructions:** Open this `.ipynb` in Colab, run the setup cell, then follow Exercises 1-5. `TODO` lines indicate the code to focus on. The notebook is deliberately runnable even before editing, so you can experiment with changes safely.

Packages: numpy scipy scikit-learn plotly matplotlib pillow imageio imageio-ffmpeg. They are normally available in Colab; if an import fails, uncomment the install cell below.

In [ ]:
# @title Optional install cell
# %pip -q install numpy scipy scikit-learn plotly matplotlib pillow imageio imageio-ffmpeg

In [ ]:
# @title Setup and plotting functions (run once; you do not need to edit)
"""Demonstration visuals for a 1D BO

Synthetic pedagogical reaction response only: not experimental data
"""
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF

SEED = 2026
LOW, HIGH = 60.0, 140.0
GRID = np.linspace(LOW, HIGH, 321)
BASELINE = 42.0
AMPLITUDE = 25.0
LENGTH_SCALE = 14.0
NOISE_STD = 2.0
INITIAL_X = np.array([65.0, 90.0, 130.0])
COLORS = {
    'mean': '#2463a0', 'band': 'rgba(67, 128, 194, 0.19)',
    'observation': '#17212e', 'proposed': '#db6f38',
    'acq': '#16846e', 'light': '#eef3f8', 'text': '#223249',
}


def true_yield(temperature):
    """Toy response with two peaks"""
    x = np.asarray(temperature, dtype=float)
    return (23.0
        + 37.0 * np.exp(-0.5 * ((x - 83.0) / 10.0)**2)
        + 54.0 * np.exp(-0.5 * ((x - 116.0) / 7.0)**2)
        + 3.0 * np.sin((x - 60.0) / 7.0))


def initial_measurements():
    rng = np.random.default_rng(SEED)
    return INITIAL_X.copy(), true_yield(INITIAL_X) + rng.normal(0, NOISE_STD, len(INITIAL_X))


def gp_model(x_obs, y_obs):
    """GP models latent f, while alpha represents known independent observation noise."""
    kernel = ConstantKernel(AMPLITUDE**2, constant_value_bounds='fixed') * RBF(
        length_scale=LENGTH_SCALE, length_scale_bounds='fixed')
    gp = GaussianProcessRegressor(
        kernel=kernel, alpha=NOISE_STD**2, optimizer=None,
        normalize_y=False, random_state=SEED)
    gp.fit(np.asarray(x_obs, dtype=float).reshape(-1, 1), np.asarray(y_obs) - BASELINE)
    return gp


def posterior(x_obs, y_obs, x_grid=GRID):
    gp = gp_model(x_obs, y_obs)
    mean_centered, std = gp.predict(np.asarray(x_grid).reshape(-1, 1), return_std=True)
    return mean_centered + BASELINE, std


def ucb(mean, std, kappa=2.0):
    return np.asarray(mean) + float(kappa) * np.asarray(std)


def choose_next(x_grid, mean, std, x_obs, kappa=2.0, exclusion_radius=0.8):
    acquisition = ucb(mean, std, kappa)
    too_close = np.min(np.abs(np.asarray(x_grid)[:, None] - np.asarray(x_obs)[None, :]), axis=1) < exclusion_radius
    allowed = np.where(too_close, -np.inf, acquisition)
    idx = int(np.argmax(allowed))
    return float(x_grid[idx]), acquisition


def generate_history(kappa=2.0, n_steps=7):
    rng = np.random.default_rng(SEED)
    x_obs = INITIAL_X.copy()
    y_obs = true_yield(x_obs) + rng.normal(0, NOISE_STD, len(x_obs))
    states = []
    for t in range(n_steps + 1):
        mu, sd = posterior(x_obs, y_obs)
        proposed, acq = choose_next(GRID, mu, sd, x_obs, kappa=kappa)
        states.append({
            'step': t, 'x': x_obs.copy(), 'y': y_obs.copy(),
            'mean': mu.copy(), 'std': sd.copy(), 'acq': acq.copy(),
            'proposed': proposed, 'kappa': float(kappa),
            'best': float(np.max(y_obs)), 'n_exp': len(x_obs),
        })
        if t < n_steps:
            new_y = float(true_yield(proposed)) + float(rng.normal(0, NOISE_STD))
            x_obs = np.append(x_obs, proposed)
            y_obs = np.append(y_obs, new_y)
    return states


def base_layout(title, height=610):
    return dict(
        template='plotly_white', title=dict(text=title, x=0.04, xanchor='left'),
        paper_bgcolor='white', plot_bgcolor='white', height=height,
        margin=dict(l=65, r=38, t=150, b=80),
        font=dict(family='Arial, sans-serif', size=14, color=COLORS['text']),
        legend=dict(orientation='h', y=1.12, x=0.0, xanchor='left'),
        hovermode='x',
    )


def band_trace(mu, sd, row_label=None):
    return go.Scatter(
        x=np.concatenate([GRID, GRID[::-1]]),
        y=np.concatenate([mu + 2*sd, (mu - 2*sd)[::-1]]),
        fill='toself', fillcolor=COLORS['band'], line=dict(color='rgba(0,0,0,0)'),
        name='GP mean +/- 2 sd', hoverinfo='skip', showlegend=row_label is None)


def mean_trace(mu):
    return go.Scatter(x=GRID, y=mu, name='GP mean', mode='lines',
                      line=dict(color=COLORS['mean'], width=3),
                      hovertemplate='T=%{x:.1f} C<br>Predicted yield=%{y:.1f}%<extra></extra>')


def data_trace(x, y):
    return go.Scatter(x=x, y=y, mode='markers', name='Measured experiments',
                      marker=dict(size=11, color=COLORS['observation'],
                                  line=dict(width=1.8, color='white')),
                      hovertemplate='T=%{x:.1f} C<br>Measured yield=%{y:.1f}%<extra></extra>')


def proposal_trace(x, mu):
    return go.Scatter(x=[x], y=[float(np.interp(x, GRID, mu))], mode='markers',
        name='Proposed next experiment', marker=dict(size=16, symbol='star',
        color=COLORS['proposed'], line=dict(width=1.3, color='white')),
        hovertemplate='Proposed T=%{x:.1f} C<extra></extra>')


def acq_trace(a):
    return go.Scatter(x=GRID, y=a, mode='lines', name='UCB acquisition',
        line=dict(color=COLORS['acq'], width=3),
        hovertemplate='T=%{x:.1f} C<br>UCB=%{y:.1f}<extra></extra>')


def acq_selected_trace(x, a):
    return go.Scatter(x=[x], y=[float(np.interp(x, GRID, a))], mode='markers',
        name='Chosen by UCB', showlegend=False,
        marker=dict(size=14, color=COLORS['proposed'], symbol='star', line=dict(color='white', width=1.3)))


def animation_buttons(frame_ms=950):
    return [dict(type='buttons', showactive=False, x=0.01, y=-0.20, xanchor='left',
                 yanchor='top', direction='left', buttons=[
        dict(label='Play', method='animate', args=[None, dict(
            frame=dict(duration=frame_ms, redraw=True), transition=dict(duration=0),
            fromcurrent=True, mode='immediate')]),
        dict(label='Pause', method='animate', args=[[None], dict(
            frame=dict(duration=0, redraw=False), transition=dict(duration=0),
            mode='immediate')]),
    ])]


def animation_slider(names, labels):
    return [dict(active=0, x=0.25, y=-0.175, len=0.75, xanchor='left',
                 currentvalue=dict(prefix='', font=dict(size=13)),
                 pad=dict(t=20, b=5), steps=[
                     dict(label=lbl, method='animate', args=[[name], dict(
                         frame=dict(duration=0, redraw=True),
                         transition=dict(duration=0), mode='immediate')])
                     for name, lbl in zip(names, labels)])]


def prior_states():
    x0, y0 = initial_measurements()
    x_small = np.linspace(LOW, HIGH, 101)
    r = np.random.default_rng(323)
    kernel = ConstantKernel(AMPLITUDE**2) * RBF(LENGTH_SCALE)
    result = []
    for n in range(4):
        if n == 0:
            mu = np.full_like(x_small, BASELINE)
            sd = np.full_like(x_small, AMPLITUDE)
            covariance = kernel(x_small[:, None])
            samples = mu[:, None] + np.linalg.cholesky(covariance + 1e-6 * np.eye(len(x_small))) @ r.normal(size=(len(x_small), 4))
        else:
            gp = gp_model(x0[:n], y0[:n])
            m, sd = gp.predict(x_small[:, None], return_std=True)
            mu = m + BASELINE
            samples = gp.sample_y(x_small[:, None], n_samples=4, random_state=100 + n) + BASELINE
        result.append(dict(n=n, xs=x0[:n], ys=y0[:n], xx=x_small, mean=mu, std=sd, samples=samples))
    return result


def prior_posterior_figure():
    states = prior_states()
    plot_colors = ['#99aec2', '#98b3cf', '#80a3c8', '#9caec8']
    def traces(s):
        xx, mu, sd = s['xx'], s['mean'], s['std']
        arr = [go.Scatter(x=np.r_[xx, xx[::-1]], y=np.r_[mu+2*sd, (mu-2*sd)[::-1]],
            fill='toself', fillcolor=COLORS['band'], line=dict(color='rgba(0,0,0,0)'),
            name='Mean +/- 2 sd', hoverinfo='skip')]
        for i in range(4):
            arr.append(go.Scatter(x=xx, y=s['samples'][:,i], mode='lines',
                line=dict(color=plot_colors[i], width=1.9, dash='dot'),
                opacity=0.75, name='Function samples' if i==0 else 'Sample',
                showlegend=(i==0)))
        arr.append(go.Scatter(x=xx, y=mu, mode='lines',
            line=dict(color=COLORS['mean'], width=3), name='GP mean'))
        arr.append(data_trace(s['xs'], s['ys']))
        return arr
    names = [f'step_{i}' for i in range(len(states))]
    labels = ['Prior', '1 point', '2 points', '3 points']
    frames = [go.Frame(name=nm, data=traces(s), traces=list(range(7)),
            layout=go.Layout(title_text=(
                'GP prior: possible functions before data' if s['n']==0 else
                f'GP posterior: conditioned on {s["n"]} observation(s)')))
            for nm,s in zip(names,states)]
    fig = go.Figure(data=traces(states[0]), frames=frames)
    layout = base_layout('GP prior: possible functions before data', 610)
    fig.update_layout(**layout, updatemenus=animation_buttons(), sliders=animation_slider(names, labels))
    fig.update_xaxes(range=[LOW, HIGH], title_text='Reaction temperature (C)', zeroline=False)
    fig.update_yaxes(range=[-45, 125], title_text='Reaction yield (%)', zeroline=False)
    return fig


def ucb_slider_figure():
    x0, y0 = initial_measurements()
    mu, sd = posterior(x0,y0)
    kappas = [0., .5, 1., 1.5, 2., 3., 4.]
    def variable_traces(k):
        x, a = choose_next(GRID, mu, sd, x0, k)
        return [proposal_trace(x,mu), acq_trace(a), acq_selected_trace(x,a)]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        vertical_spacing=0.14, row_heights=[0.58,0.42],
        subplot_titles=('GP posterior (fixed data)', 'UCB acquisition (change kappa)'))
    fig.add_trace(band_trace(mu,sd), row=1,col=1)
    fig.add_trace(mean_trace(mu),row=1,col=1)
    fig.add_trace(data_trace(x0,y0),row=1,col=1)
    for tr,row in zip(variable_traces(kappas[0]),[1,2,2]):
        fig.add_trace(tr,row=row,col=1)
    names = [f'k_{i}' for i in range(len(kappas))]
    frames = [go.Frame(name=nm, data=variable_traces(k), traces=[3,4,5],
        layout=go.Layout(title_text=f'Exploration vs exploitation: kappa = {k:g}'))
        for nm,k in zip(names,kappas)]
    fig.frames=frames
    fig.update_layout(**base_layout('Exploration vs exploitation: kappa = 0', 780),
        updatemenus=animation_buttons(frame_ms=1050),
        sliders=animation_slider(names,[f'{k:g}' for k in kappas]))
    fig.update_xaxes(range=[LOW,HIGH], row=1,col=1)
    fig.update_xaxes(range=[LOW,HIGH], title_text='Reaction temperature (C)', row=2,col=1)
    fig.update_yaxes(title_text='Yield (%)', range=[-5,105], row=1,col=1)
    fig.update_yaxes(title_text='UCB score', range=[0,145], row=2,col=1)
    return fig


def bo_loop_figure(states=None):
    if states is None:
        states = generate_history()
    def traces(s):
        return [band_trace(s['mean'], s['std']), mean_trace(s['mean']),
            data_trace(s['x'],s['y']), proposal_trace(s['proposed'],s['mean']),
            acq_trace(s['acq']), acq_selected_trace(s['proposed'],s['acq'])]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        vertical_spacing=0.12, row_heights=[0.63,0.37],
        subplot_titles=('Surrogate: learned mean and uncertainty',
                        'Decision rule: Upper Confidence Bound'))
    for tr,row in zip(traces(states[0]),[1,1,1,1,2,2]):
        fig.add_trace(tr,row=row,col=1)
    names=[f'iteration_{s["step"]}' for s in states]
    frames=[]
    for name,s in zip(names,states):
        frames.append(go.Frame(name=name, data=traces(s), traces=list(range(6)),
            layout=go.Layout(title_text=(
                f'BO iteration {s["step"]}  |  {s["n_exp"]} observations  |  '
                f'best measured yield = {s["best"]:.1f}%'))))
    fig.frames=frames
    fig.update_layout(**base_layout('BO iteration 0  |  3 observations', 790),
        updatemenus=animation_buttons(frame_ms=1400),
        sliders=animation_slider(names,[str(s['step']) for s in states]))
    fig.update_xaxes(range=[LOW,HIGH], row=1,col=1)
    fig.update_xaxes(range=[LOW,HIGH], title_text='Reaction temperature (C)', row=2,col=1)
    fig.update_yaxes(title_text='Yield (%)',range=[-5,105],row=1,col=1)
    fig.update_yaxes(title_text='UCB score',range=[0,140],row=2,col=1)
    return fig


def render_static_state(s, title=None, reveal_truth=False, figsize=(13.33,7.5)):
    fig,(ax1,ax2)=plt.subplots(2,1,figsize=figsize,sharex=True,
        gridspec_kw={'height_ratios':[1.6,1]},layout='constrained')
    fig.patch.set_facecolor('white')
    mu,sd=s['mean'],s['std']
    ax1.fill_between(GRID,mu-2*sd,mu+2*sd,color='#7eacd8',alpha=.23,label='GP mean +/- 2 sd')
    ax1.plot(GRID,mu,color=COLORS['mean'],lw=2.6,label='GP mean')
    if reveal_truth:
        ax1.plot(GRID,true_yield(GRID),color='#7d8793',lw=1.9,ls='--',label='True response (revealed)')
    ax1.scatter(s['x'],s['y'],s=55,color=COLORS['observation'],edgecolor='white',linewidth=1.1,zorder=5,label='Measured')
    x=s['proposed']
    ax1.scatter([x],[np.interp(x,GRID,mu)],marker='*',s=260,color=COLORS['proposed'],edgecolor='white',linewidth=1.2,zorder=7,label='Next proposal')
    ax1.axvline(x,color=COLORS['proposed'],ls='--',lw=1.25,alpha=.75)
    ax1.set_ylim(-5,105)
    ax1.set_ylabel('Reaction yield (%)',fontsize=12)
    ax1.set_title('Probabilistic surrogate',loc='left',fontsize=15,fontweight='bold',pad=11)
    ax1.legend(loc='upper left',ncol=3,fontsize=9.5,frameon=False)
    ax2.plot(GRID,s['acq'],color=COLORS['acq'],lw=2.8)
    ax2.scatter([x],[np.interp(x,GRID,s['acq'])],marker='*',s=220,color=COLORS['proposed'],edgecolor='white',linewidth=1.1,zorder=6)
    ax2.axvline(x,color=COLORS['proposed'],ls='--',lw=1.25,alpha=.75)
    ax2.set_ylim(0,140)
    ax2.set_ylabel('UCB acquisition',fontsize=12)
    ax2.set_title('Decision rule: mean + 2 x standard deviation',loc='left',fontsize=14,fontweight='bold',pad=10)
    ax2.set_xlabel('Reaction temperature (C)',fontsize=12)
    ax2.xaxis.set_major_locator(MultipleLocator(10))
    for ax in (ax1,ax2):
        ax.set_xlim(LOW,HIGH)
        ax.grid(True,color='#c4d0dd',alpha=.33)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    fig.suptitle(title if title else (
        f'BO iteration {s["step"]} | {s["n_exp"]} experiments | '
        f'best observed yield {s["best"]:.1f}%'),
        fontsize=19,fontweight='bold',color=COLORS['text'])
    return fig


def render_prior_state(s):
    fig,ax=plt.subplots(figsize=(12.8,7.2),layout='constrained')
    xx,mu,sd=s['xx'],s['mean'],s['std']
    ax.fill_between(xx,mu-2*sd,mu+2*sd,color='#7eacd8',alpha=.23,label='Mean +/- 2 sd')
    for i in range(s['samples'].shape[1]):
        ax.plot(xx,s['samples'][:,i],color='#8eaac9',ls='--',alpha=.75,lw=1.5,
                label='Possible function' if i==0 else None)
    ax.plot(xx,mu,color=COLORS['mean'],lw=3,label='GP mean')
    if s['n']:
        ax.scatter(s['xs'],s['ys'],s=100,color=COLORS['observation'],
                   edgecolor='white',linewidth=1.4,zorder=5,label='Observed yield')
    ax.set(xlim=(LOW,HIGH),ylim=(-45,125),xlabel='Reaction temperature (C)',ylabel='Response (%)')
    ax.grid(alpha=.22)
    ax.legend(loc='upper right',frameon=False,ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    title=('GP prior: possible functions (no observations)' if s['n']==0
           else f'GP posterior: conditioned on {s["n"]} observation(s)')
    fig.suptitle(title,fontsize=19,fontweight='bold',color=COLORS['text'])
    return fig


def render_ucb_state(kappa):
    x0,y0=initial_measurements()
    mu,sd=posterior(x0,y0)
    proposed,acq=choose_next(GRID,mu,sd,x0,kappa)
    fig,(ax1,ax2)=plt.subplots(2,1,figsize=(12.8,7.2),sharex=True,
                gridspec_kw={'height_ratios':[1.6,1]},layout='constrained')
    ax1.fill_between(GRID,mu-2*sd,mu+2*sd,color='#7eacd8',alpha=.23,label='GP mean +/- 2 sd')
    ax1.plot(GRID,mu,color=COLORS['mean'],lw=2.6,label='GP mean')
    ax1.scatter(x0,y0,s=65,color=COLORS['observation'],edgecolor='white',lw=1,zorder=5,label='Measurements')
    ax1.scatter([proposed],[np.interp(proposed,GRID,mu)],s=250,marker='*',color=COLORS['proposed'],zorder=7,label='Proposal')
    ax1.axvline(proposed,ls='--',color=COLORS['proposed'],alpha=.7)
    ax1.set(ylim=(-5,105),ylabel='Yield (%)')
    ax1.legend(frameon=False,ncol=4,fontsize=9.5)
    ax1.set_title('Fixed GP posterior',loc='left',fontsize=14,fontweight='bold')
    ax2.plot(GRID,acq,color=COLORS['acq'],lw=2.8)
    ax2.scatter([proposed],[np.interp(proposed,GRID,acq)],s=225,marker='*',color=COLORS['proposed'],zorder=7)
    ax2.axvline(proposed,ls='--',color=COLORS['proposed'],alpha=.7)
    ax2.set(ylim=(0,145),ylabel='UCB score',xlabel='Reaction temperature (C)')
    ax2.set_title(f'Acquisition: mean + {kappa:g} x standard deviation',loc='left',fontsize=14,fontweight='bold')
    for ax in (ax1,ax2):
        ax.set_xlim(LOW,HIGH)
        ax.grid(alpha=.22)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    fig.suptitle(f'Exploration vs exploitation | kappa = {kappa:g} | next T = {proposed:.1f} C',
                 fontsize=18,fontweight='bold',color=COLORS['text'])
    return fig


def save_gif_mp4_from_figures(figure_list,folder,stem):
    import io
    from PIL import Image
    import imageio
    folder=Path(folder)
    frames=[]
    for fig in figure_list:
        buf=io.BytesIO()
        fig.savefig(buf,format='png',dpi=100,facecolor='white')
        plt.close(fig)
        buf.seek(0)
        frames.append(np.asarray(Image.open(buf).convert('RGB')))
    with imageio.get_writer(str(folder/f'{stem}.mp4'),fps=2,codec='libx264',quality=7,
                            macro_block_size=16) as writer:
        for frame in frames:
            for _ in range(3):
                writer.append_data(frame)
    gifs=[Image.fromarray(im).resize((960,540),Image.Resampling.LANCZOS) for im in frames]
    gifs[0].save(folder/f'{stem}.gif',save_all=True,append_images=gifs[1:],
                 duration=1500,loop=0,optimize=False)


def export_assets(folder, make_video=True):
    path=Path(folder)
    path.mkdir(parents=True,exist_ok=True)
    x0,y0=initial_measurements()
    states=generate_history(kappa=2.0,n_steps=7)
    figs=[('01_gp_prior_to_posterior.html',prior_posterior_figure()),
          ('02_ucb_exploration_slider.html',ucb_slider_figure()),
          ('03_bayesian_optimisation_loop.html',bo_loop_figure(states))]
    for name,fig in figs:
        fig.write_html(str(path/name),include_plotlyjs=True,full_html=True,
                       config={'displaylogo':False,'responsive':True})
    for idx in [0,4,7]:
        fig=render_static_state(states[idx])
        fig.savefig(path/f'bo_iteration_{idx}.png',dpi=155,facecolor='white')
        plt.close(fig)
    fig=render_static_state(states[-1],reveal_truth=True,title='BO: reveal the hidden synthetic response')
    fig.savefig(path/'bo_reveal_true_response.png',dpi=155,facecolor='white')
    plt.close(fig)
    if make_video:
        import io
        from PIL import Image
        import imageio.v2 as imageio
        frames=[]
        for s in states:
            fig=render_static_state(s,figsize=(12.8,7.2))
            buf=io.BytesIO()
            fig.savefig(buf,format='png',dpi=100,facecolor='white')
            plt.close(fig)
            buf.seek(0)
            frames.append(np.asarray(Image.open(buf).convert('RGB')))
        # Long dwell on each scientific step; no tweening between fitted GPs.
        import imageio
        with imageio.get_writer(str(path/'bo_iteration_animation.mp4'),fps=2,codec='libx264',quality=7,
                                macro_block_size=16) as writer:
            for im in frames:
                for _ in range(3):
                    writer.append_data(im)
        from PIL import Image
        gif_frames=[Image.fromarray(im).resize((960,540),Image.Resampling.LANCZOS) for im in frames]
        gif_frames[0].save(path/'bo_iteration_animation.gif',save_all=True,
                           append_images=gif_frames[1:],duration=1500,loop=0,optimize=False)
    if make_video:
        save_gif_mp4_from_figures([render_prior_state(s) for s in prior_states()],
                                  path, 'gp_prior_to_posterior_animation')
        save_gif_mp4_from_figures([render_ucb_state(k) for k in [0.,.5,1.,1.5,2.,3.,4.]],
                                  path, 'ucb_kappa_animation')
    return states

## Exercise 1 (5 min): inspect your starting experiments
Plot three measurements taken at 65, 90 and 130 C. **Before running:** Which observed temperature do you expect to have the greatest yield? Does that reveal the optimum?

In [ ]:
x_obs, y_obs = initial_measurements()
fig, ax = plt.subplots(figsize=(10,4))
ax.scatter(x_obs, y_obs, s=100, c=COLORS['observation'], zorder=3)
ax.set(xlim=(LOW,HIGH),ylim=(0,100),xlabel='Reaction temperature (C)',ylabel='Measured yield (%)',
       title='Three initial experiments (true response intentionally hidden)')
ax.grid(alpha=.25)
plt.show()
print('Best observed starting temperature:', x_obs[np.argmax(y_obs)],'C')

## Exercise 2 (10 min): fit and inspect a Gaussian process
The function `posterior()` implements a GP with RBF covariance and known noise variance. We plot the posterior mean and a mean +/- 2 sd uncertainty band.

**Tasks:**
1. Identify high-uncertainty regions
1. Change `LENGTH_SCALE` in the setup cell from 14 to 7 or 28, rerun it, and compare the posterior
1. **Restore 14 for the remaining exercises**

In [ ]:
mu, sigma = posterior(x_obs, y_obs)
assert mu.shape == GRID.shape == sigma.shape
assert np.all(sigma >= 0)
fig, ax = plt.subplots(figsize=(11,4.4))
ax.fill_between(GRID, mu-2*sigma, mu+2*sigma, color='#7eacd8', alpha=.25, label='Mean +/- 2 sd')
ax.plot(GRID,mu,color=COLORS['mean'],lw=2.6,label='GP mean')
ax.scatter(x_obs,y_obs,s=75,c=COLORS['observation'],label='Measurements',zorder=5)
ax.set(xlim=(LOW,HIGH),ylim=(-5,100),xlabel='Temperature (C)',ylabel='Predicted yield (%)')
ax.legend(frameon=False)
ax.grid(alpha=.2)
plt.show()

## Exercise 3 (7 min): implement the UCB decision rule
The UCB formula is:

$$a_{\mathrm{UCB}}(x)=\mu(x)+\kappa \sigma(x)$$

**TODO:**
1. Rewrite the return statement of `student_ucb()` from memory, using the equation above
1. Test $\kappa$ = 0, 1, 2 and 4
1. Explain why the selected temperature can move even when the GP is unchanged

In [ ]:
def student_ucb(mean, std, kappa):
    # TODO: Write the UCB equation here from the formula above
    ## here
    return

assert np.allclose(student_ucb(mu, sigma, 0), mu)
assert np.allclose(student_ucb(mu, sigma, 2), mu + 2*sigma)

for kappa in [0,1,2,4]:
    acquisition = student_ucb(mu,sigma,kappa)
    next_idx = int(np.argmax(acquisition))
    print(f'kappa={kappa}: candidate temperature = {GRID[next_idx]:.2f} C')

# Interactive: pause and inspect a kappa value before returning to the code.
ucb_slider_figure().show()

## Exercise 4 (8 min): implement the select / measure / update loop
Each iteration: **fit GP -> maximize UCB -> measure -> add a new observation**.

**TODO:**
1. Locate the four key lines in the cell and explain each
1. Try changing `kappa` or `n_steps` to see the impact on the trajectory
1. The synthetic `true_yield()` function stands in for the robot / experimental measurement

In [ ]:
kappa = 2.0  # TODO: Try 0.0, 1.0 and 4.0 later.
n_steps = 7
rng = np.random.default_rng(101)
x_run, y_run = initial_measurements()
record = [float(np.max(y_run))]

for t in range(n_steps):
    mu_t, sd_t = posterior(x_run, y_run)
    x_next, acquisition = choose_next(GRID, mu_t, sd_t, x_run, kappa)
    y_next = float(true_yield(x_next) + rng.normal(0, NOISE_STD))
    x_run = np.append(x_run, x_next)
    y_run = np.append(y_run, y_next)
    record.append(float(np.max(y_run)))
    print(f'Experiment {t+1:2d}: T={x_next:6.2f} C | yield={y_next:5.1f}% | best={record[-1]:5.1f}%')

assert len(x_run) == len(INITIAL_X) + n_steps
assert np.all(np.diff(record) >= -1e-10), 'Best observed result should not decrease.'

fig, ax = plt.subplots(figsize=(9,4))
ax.step(np.arange(len(record)),record,where='post',color=COLORS['acq'],lw=2.7)
ax.scatter(np.arange(len(record)),record,color=COLORS['acq'])
ax.set(xlabel='Additional experiments after initial data',ylabel='Best observed yield (%)',
       title='Performance of your sequential BO run')
ax.grid(alpha=.2)
plt.show()

## Exercise 5 (5 min): compare policies
Generate interactive BO trajectories and compare several kappa settings.

**Question:** Is a setting that looks best in one noisy simulation guaranteed to outperform the others?

**Optional extension:** Implement a random-search baseline using the same evaluation budget and multiple random seeds.

In [ ]:
# Reference animation with the reproducible seed used in the slides
bo_loop_figure(generate_history(kappa=2.0, n_steps=7)).show()

for k in [0.0, 1.0, 2.0, 4.0]:
    h=generate_history(kappa=k,n_steps=7)
    print(f'kappa={k:3.1f} | best measured yield after seven iterations = {h[-1]["best"]:5.1f}%')

### Finish-up: discuss with a neighbour
1. Why is the maximum GP mean not always the best location for the *next* experiment?
2. Where in the code is the exploration / exploitation trade-off controlled?
3. What does the posterior uncertainty band mean? What does it *not* mean?
4. How would this exercise change for five chemical variables and eight experiments per batch?

**Further note:** This synthetic example uses fixed kernel hyperparameters and known Gaussian measurement noise. Real experimental BO requires attention to model fitting, constraints, mixed categorical/continuous variables, failure handling and batch selection.